# M14 · AI 自動經營(動手筆記本)

> 在 Codespaces 一格一格按 ▶ 跑。**純規則,不需金鑰也能整本跑完。**

**M12 vs M14**:M12 三角色看資料、**給建議**,人決定;M14 進化成 agent **自己動手顧店** —— 偵測狀況 → 自動止血 → 低風險自己處置、高風險升級給人(human-on-the-loop)。

你做的事:看事件 → 看護欄規則 → 跑自動經營 → 改規則再跑。

## 1. 今天進來的狀況(事件流)
看看一個營業日會冒出哪些狀況 👇

In [ ]:
import csv
events = list(csv.DictReader(open('events_sample.csv', encoding='utf-8')))
for e in events: print(f"[{e['time']}] {e['kind']:12s} {e['detail']}")

## 2. 護欄:誰能自動做、誰要升級給人

風險判斷**故意用規則**(寫在 `src/rules.py`),不交給 AI —— 這樣紅線是你定的、可稽核。
每個狀況都兩段:**① 止血一律自動;② 後續**低風險自動、高風險升級給人。

In [ ]:
from src.rules import handle
for e in events:
    stop, follow, risk, auto = handle(e['kind'])
    route = '自動處置' if auto else '⏫ 升級給人'
    print(f"{e['detail']:22s} 止血:{stop:14s} 後續:{follow:14s} [{risk}] → {route}")

## 3. 跑一次「AI 自動經營」
讓 agent 自己跑一遍,看自動經營日誌 + 升級佇列 👇

In [ ]:
import json
log, queue = [], []
print('===== AI 自動經營日誌 =====')
for e in events:
    stop, follow, risk, auto = handle(e['kind'])
    print(f"\n[{e['time']}] 偵測:{e['detail']}")
    print(f"   ① 止血(自動):{stop}  ✓")
    if auto:
        print(f"   ② {follow}(低風險,自動):已自動處理  ✓"); log.append(e['detail'])
    else:
        print(f"   ② {follow}(高風險)→ ⏫ 升級給人核准"); queue.append({'event': e['detail'], 'needs_human': follow})
print(f"\n共 {len(events)} 件:自動 {len(log)} 件、升級給人 {len(queue)} 件")
assert len(log) >= 1 and len(queue) >= 1, '應同時有自動與升級'
print('✓ human-on-the-loop:小事自動、大事升級')

## 4. 換你改規則(這就是「設紅線」)

打開 `src/rules.py`,把 `FOLLOWUP` 裡某個狀況的風險從 `low` 改成 `high`(或反過來),**重跑第 3 格** —— 那個狀況就會改走「自動」或「升級」。

> 這一步就是 M14 的核心:**AI 自己顧店,但紅線是你劃的。**

## 一句話帶走

> **M12 給建議、人決定;M14 自己動手、人設規則守紅線。**
> 出狀況 agent 自動止血、自動處理小事,只有高風險才來敲你 —— 這才是 AI 自動經營。

想看它動起來?打開 `shop-showcase/`,右下角切到 **M14 自動經營**,看 agent 自己跑。